# Evaluation

Den här notebooken utvärderar projektets metoder för
kompetensidentifiering.

Baseline utvärderas dels mot endast taxonomikompetenser,
dels mot samtliga manuellt annoterade kompetenser.

EntityRuler utvärderas mot samtliga annoterade kompetenser.

Metoderna jämförs med precision, recall och F1-score
samt genom felanalys av false positives och false negatives.

Test-setet har inte använts vid utvecklingen av regler,
synonymer eller modeller.

**WORKFLOW** :
1. Imports
2. Läser in test_set
3. Läser in manuella annotationer
4. Skapar gold_all_raw och gold_taxonomy_raw
5. Kontrollerar ID:n
6. Importerar baseline + EntityRuler från models.py
7. Kör båda metoderna på samma test-set
8. Normaliserar gold-standard
9. Skapar predictions-dictionaries
10. Beräknar precision / recall / F1
11. Sammanställer resultat
12. Felanalys
13. Kort slutsats

In [ ]:
import json
import re
import pandas as pd

In [ ]:
# läser in orginalets test-set
with open("../data/test_set.json", "r", encoding="utf-8") as file:
    test_ads = json.load(file)

test_df = pd.DataFrame(test_ads)

test_df.head()

In [ ]:
print("Antal annonser i test-set:", len(test_df))
print("Kolumner:", test_df.columns.tolist())

In [ ]:
# läser in det annoterade labelstudios test-set
with open(
    "../data/annoterad_test_data.json",
    "r",
    encoding="utf-8"
) as file:
    annotated_data = json.load(file)

In [ ]:
# undersöker strukturen på labelstudios data
print(type(annotated_data))
print("Antal poster:", len(annotated_data))

In [ ]:
annotated_data[0]

In [ ]:
annotated_data[0].keys()

In [ ]:
for key, value in annotated_data[0].items():
    print(key, ":", type(value))

In [ ]:
# bygger två gold-standard:
# 1. alla annoterade skills
# 2. endast skills märkta som Taxonomi

def extract_gold_annotations(annotated_data):
    gold_all = {}
    gold_taxonomy = {}

    for task in annotated_data:

        ad_id = str(task["data"]["id"])

        all_skills = set()
        taxonomy_skills = set()

        for annotation in task.get("annotations", []):

            for result in annotation.get("result", []):

                value = result.get("value", {})

                text = value.get("text")
                labels = value.get("labels", [])

                if not text:
                    continue

                # Alla annoterade skills
                if (
                    "Taxonomi" in labels
                    or "New skill" in labels
                ):
                    all_skills.add(text)

                # Endast skills från taxonomin
                if "Taxonomi" in labels:
                    taxonomy_skills.add(text)

        gold_all[ad_id] = all_skills
        gold_taxonomy[ad_id] = taxonomy_skills

    return gold_all, gold_taxonomy

In [ ]:
gold_all_raw, gold_taxonomy_raw = extract_gold_annotations(
    annotated_data
)

In [ ]:
print("Exempel - alla skills:")
print(list(gold_all_raw.items())[:3])

print("\nExempel - endast Taxonomi:")
print(list(gold_taxonomy_raw.items())[:3])

In [ ]:
# kollar upp att ID:n matchar mellan test-set och gold standard

test_ids = set(
    test_df["id"].astype(str)
)

gold_ids = set(
    gold_all_raw.keys()
)

print("Test-set IDs:", len(test_ids))
print("Gold IDs:", len(gold_ids))

print(
    "Saknas i gold:",
    len(test_ids - gold_ids)
)

print(
    "Saknas i test_set:",
    len(gold_ids - test_ids)
)

In [ ]:
test_df[
    ["id", "baseline_skills"]
].head()

In [ ]:
# gör dictionary
baseline_predictions = {
    str(row["id"]): set(row["baseline_skills"])
    for _, row in test_df.iterrows()
}

gold_all
        = facit med Taxonomi + New skill

gold_taxonomy
        = facit med endast Taxonomi

baseline_predictions
        = Baselinens svar

entityruler_predictions
        = EntityRulers svar

### Baseline vs endast Taxonomi

### Baseline vs alla annoterade skills 

### EntityRuler vs alla annoterade skills

### Samlade resultaten i en DataFrame

### Felanalys för baseline

### Felanalys för EntityRuler

## Precision, recall och F1-score

Utvärderingen görs på det manuellt annoterade test-setet.

**Precision** visar hur stor andel av de kompetenser som metoden
identifierade som faktiskt var korrekta.

**Recall** visar hur stor andel av de manuellt annoterade
kompetenserna som metoden lyckades hitta.

**F1-score** kombinerar precision och recall till ett gemensamt mått.

Baseline utvärderas både mot endast taxonomikompetenser och mot
samtliga annoterade kompetenser. EntityRuler utvärderas mot samtliga
annoterade kompetenser.

## Error analysis

Förutom precision, recall och F1-score genomförs en felanalys.

**False positive (FP)** innebär att metoden identifierade en kompetens
som inte fanns i det manuella facit.

**False negative (FN)** innebär att en kompetens fanns i det manuella
facit men inte identifierades av metoden.

Felanalysen används för att förstå metodernas styrkor och
begränsningar.

Test-setet används inte för att skapa eller justera nya regler efter
den slutliga utvärderingen.